# 🚀 X-Ray Object Detection — Standalone Colab Notebook
This notebook contains the **entire** project consolidated into a single file.

**Instructions:**
1. Go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Upload your raw dataset ZIP to `/content/data/raw/` if it's not already there.
3. Click **Run All**.


In [ ]:
!pip install -q ultralytics pyyaml pandas opencv-python matplotlib tqdm
import os
import sys
from pathlib import Path
import shutil
import glob
import time
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from collections import Counter
import yaml
import random
from tqdm.auto import tqdm



In [ ]:
# --- Google Colab / Local Paths ---
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = '/content/drive/MyDrive/Object-Detection-in-Luggage-Scanner-for-Security'
else:
    WORKSPACE = os.path.abspath('.')

DATA_ROOT = os.path.join(WORKSPACE, 'data')
RAW_DIR = os.path.join(DATA_ROOT, 'raw')
PROCESSED_DIR = os.path.join(DATA_ROOT, 'processed')
CHECKPOINT_DIR = os.path.join(WORKSPACE, 'checkpoints')
RESULTS_DIR = os.path.join(WORKSPACE, 'results')

for d in [RAW_DIR, PROCESSED_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Workspace set to: {WORKSPACE}")


In [ ]:
# --- Master Configuration ---
config = {'project': {'name': 'xray-object-detection', 'version': '0.1.0'}, 'paths': {'data_root': 'data', 'raw_data': 'data/raw', 'processed_data': 'data/processed', 'annotations': 'data/annotations', 'checkpoints': 'checkpoints', 'results': 'results', 'drive_root': '/content/drive/MyDrive/xray_detection', 'drive_data': '/content/drive/MyDrive/xray_detection/data', 'drive_checkpoints': '/content/drive/MyDrive/xray_detection/checkpoints'}, 'dataset': {'name': 'sixray', 'image_size': 640, 'num_workers': 4, 'pin_memory': True, 'train_ratio': 0.7, 'val_ratio': 0.15, 'test_ratio': 0.15, 'threat_classes': ['gun', 'knife', 'wrench', 'pliers', 'scissors'], 'benign_classes': ['non-threat']}, 'property_schema': {'num_properties': 11, 'properties': [{'name': 'edge_sharpness', 'type': 'float', 'range': [0.0, 1.0], 'description': 'Canny edge detection density on object mask'}, {'name': 'length_to_width_ratio', 'type': 'float', 'range': [0.0, 10.0], 'description': 'Bounding box aspect ratio'}, {'name': 'symmetry_score', 'type': 'float', 'range': [0.0, 1.0], 'description': 'Axis-aligned pixel distribution comparison'}, {'name': 'curvature_index', 'type': 'float', 'range': [0.0, 1.0], 'description': 'Contour curvature analysis'}, {'name': 'approximate_volume', 'type': 'float', 'range': [0.0, 1.0], 'description': 'Pixel area × opacity-estimated depth (normalized)'}, {'name': 'material_category', 'type': 'categorical', 'num_classes': 4, 'classes': ['organic', 'metallic', 'mixed', 'opaque'], 'description': 'Dual-energy HE/LE channel analysis'}, {'name': 'avg_absorption_intensity', 'type': 'float', 'range': [0.0, 1.0], 'description': 'Mean pixel value in grayscale channel'}, {'name': 'material_homogeneity', 'type': 'float', 'range': [0.0, 1.0], 'description': 'Std deviation of absorption across object'}, {'name': 'density_level', 'type': 'float', 'range': [0.0, 1.0], 'description': 'Weighted HE+LE intensity ratio'}, {'name': 'sharp_edge_count', 'type': 'integer', 'range': [0, 20], 'description': 'Harris / Shi-Tomasi corner detection count'}, {'name': 'occlusion_score', 'type': 'float', 'range': [0.0, 1.0], 'description': 'Fraction of bbox covered by overlapping objects'}]}, 'preprocessing': {'bilateral': {'d': 9, 'sigma_color': 75, 'sigma_space': 75}, 'clahe': {'clip_limit': 2.0, 'tile_grid_size': [8, 8]}, 'target_size': 640, 'num_channels': 4}, 'augmentation': {'horizontal_flip_prob': 0.5, 'rotation_limit': 30, 'scale_range': [0.8, 1.2], 'elastic_transform': True, 'brightness_contrast_limit': 0.15, 'gauss_noise_var_limit': [10.0, 50.0], 'random_gamma': True, 'cutmix_prob': 0.3}, 'model1': {'backbone': 'yolov8m', 'pretrained': True, 'input_channels': 4, 'property_head': {'hidden_dims': [512, 256], 'num_outputs': 10, 'activation': 'relu', 'batch_norm': True, 'dropout': 0.1}, 'material_branch': {'enabled': True, 'num_classes': 4}, 'stage1': {'epochs': 100, 'batch_size': 16, 'learning_rate': 0.01, 'optimizer': 'SGD', 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3, 'scheduler': 'cosine'}, 'stage2': {'epochs': 50, 'batch_size': 16, 'learning_rate': 0.001, 'optimizer': 'Adam', 'weight_decay': 0.0001, 'freeze_backbone': True, 'scheduler': 'cosine'}, 'loss': {'uncertainty_weighting': True, 'gradient_clip_max_norm': 1.0, 'property_loss': 'mse', 'material_loss': 'cross_entropy', 'focal_loss': {'gamma': 2.0, 'alpha': 0.25}}}, 'evaluation': {'detection': {'map50_target': 0.9, 'map50_95_target': 0.7}, 'property': {'mae_target': 0.1, 'rmse_target': 0.15}, 'material': {'accuracy_target': 0.92}}, 'wandb': {'enabled': False, 'project': 'xray-detection', 'entity': None}}



### Configuration Utils


In [ ]:
"""
Configuration loader for the X-Ray Object Detection project.
Loads YAML config files and provides easy access to parameters.
"""

import os
import yaml
from pathlib import Path


def load_config(config_path: str = None) -> dict:
    """
    Load configuration from a YAML file.

    Args:
        config_path: Path to config YAML file. If None, loads configs/default.yaml
                     relative to the project root.

    Returns:
        Dictionary with all configuration parameters.
    """
    if config_path is None:
        # Find project root (directory containing 'configs/')
        project_root = _find_project_root()
        config_path = os.path.join(project_root, "configs", "default.yaml")

    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    return config


def _find_project_root() -> str:
    """
    Find the project root by looking for the 'configs' directory.
    Walks up from the current file's location.
    """
    current = Path(__file__).resolve().parent
    for _ in range(5):  # Walk up at most 5 levels
        if (current / "configs").exists():
            return str(current)
        current = current.parent

    # Fallback: use current working directory
    return os.getcwd()


def get_property_names(config: dict) -> list:
    """Get list of property names from the config schema."""
    return [p["name"] for p in config["property_schema"]["properties"]]


def get_property_ranges(config: dict) -> dict:
    """Get property name -> (min, max) range mapping."""
    ranges = {}
    for p in config["property_schema"]["properties"]:
        if "range" in p:
            ranges[p["name"]] = tuple(p["range"])
    return ranges


def resolve_paths(config: dict, base_dir: str = None, use_colab: bool = False) -> dict:
    """
    Resolve relative paths in config to absolute paths.

    Args:
        config: Configuration dictionary.
        base_dir: Base directory for resolving relative paths.
        use_colab: If True, use Google Drive paths from config.

    Returns:
        Config with resolved paths.
    """
    if base_dir is None:
        base_dir = _find_project_root()

    paths = config.get("paths", {})

    if use_colab:
        # In Colab, use Drive-based paths for data and checkpoints
        resolved = {
            "data_root": paths.get("drive_data", "/content/drive/MyDrive/xray_detection/data"),
            "raw_data": os.path.join(paths.get("drive_data", ""), "raw"),
            "processed_data": os.path.join(paths.get("drive_data", ""), "processed"),
            "annotations": os.path.join(paths.get("drive_data", ""), "annotations"),
            "checkpoints": paths.get("drive_checkpoints", "/content/drive/MyDrive/xray_detection/checkpoints"),
            "results": os.path.join(paths.get("drive_root", ""), "results"),
        }
    else:
        # Local development
        resolved = {}
        for key in ["data_root", "raw_data", "processed_data", "annotations", "checkpoints", "results"]:
            if key in paths:
                p = paths[key]
                if not os.path.isabs(p):
                    p = os.path.join(base_dir, p)
                resolved[key] = p

    config["resolved_paths"] = resolved
    return config



### Segmentation & Preprocessing


In [ ]:
"""
Three-stage segmentation refinement for X-ray images.

Stage 1: Saliency detection (simplified — Otsu + morphological operations)
Stage 2: Mask R-CNN coarse masks (via detectron2, optional)
Stage 3: GrabCut refinement (cv2.grabCut)

Note: Full detectron2/U2-Net requires GPU and specific installations.
      This module provides fallback methods for CPU-only environments.
"""

import cv2
import numpy as np
from typing import List, Optional, Tuple


class SegmentationRefinement:
    """
    Three-stage segmentation refinement pipeline.

    For production use, stages 1-2 should use U2-Net and Mask R-CNN respectively.
    This implementation provides OpenCV-based fallbacks for environments without
    detectron2 installed (e.g., local development without GPU).

    Usage:
        segmentor = SegmentationRefinement()
        masks = segmentor.segment(image, bboxes)
    """

    def __init__(
        self,
        use_detectron2: bool = False,
        detectron2_config: Optional[str] = None,
        detectron2_weights: Optional[str] = None,
        grabcut_iterations: int = 5,
    ):
        self.use_detectron2 = use_detectron2
        self.grabcut_iterations = grabcut_iterations
        self.detector = None

        if use_detectron2:
            self._init_detectron2(detectron2_config, detectron2_weights)

    def _init_detectron2(self, config_path: Optional[str], weights_path: Optional[str]):
        """Initialize Mask R-CNN with detectron2."""
        try:
            import detectron2
            from detectron2.config import get_cfg
            from detectron2.engine import DefaultPredictor
            from detectron2 import model_zoo

            cfg = get_cfg()
            if config_path:
                cfg.merge_from_file(config_path)
            else:
                cfg.merge_from_file(model_zoo.get_config_file(
                    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
                ))
            if weights_path:
                cfg.MODEL.WEIGHTS = weights_path
            else:
                cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
                    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
                )
            cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
            cfg.MODEL.DEVICE = "cuda" if self._has_gpu() else "cpu"

            self.detector = DefaultPredictor(cfg)
            print("✓ Detectron2 Mask R-CNN initialized")
        except ImportError:
            print("⚠ detectron2 not installed, using OpenCV fallback")
            self.use_detectron2 = False

    @staticmethod
    def _has_gpu() -> bool:
        try:
            import torch
            return torch.cuda.is_available()
        except ImportError:
            return False

    def segment(
        self,
        image: np.ndarray,
        bboxes: List[List[int]],
    ) -> List[np.ndarray]:
        """
        Run full three-stage segmentation pipeline per bounding box.

        Args:
            image: Input image (H, W) grayscale or (H, W, 3) color.
            bboxes: List of [x1, y1, x2, y2] bounding boxes.

        Returns:
            List of binary masks (H, W) for each bbox.
        """
        masks = []

        for bbox in bboxes:
            # Stage 1: Saliency-based initial segmentation
            saliency_mask = self._saliency_segmentation(image, bbox)

            # Stage 2: Mask R-CNN (if available) or use saliency result
            if self.use_detectron2 and self.detector is not None:
                rcnn_mask = self._maskrcnn_segmentation(image, bbox)
                # Combine: use RCNN if available, else saliency
                coarse_mask = rcnn_mask if rcnn_mask is not None else saliency_mask
            else:
                coarse_mask = saliency_mask

            # Stage 3: GrabCut refinement
            refined_mask = self._grabcut_refinement(image, bbox, coarse_mask)

            masks.append(refined_mask)

        return masks

    def _saliency_segmentation(
        self, image: np.ndarray, bbox: List[int]
    ) -> np.ndarray:
        """
        Stage 1: Saliency-based foreground detection.
        Uses Otsu thresholding + morphological operations as U2-Net fallback.
        """
        h, w = image.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)

        x1, y1, x2, y2 = [int(c) for c in bbox]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)

        if x2 <= x1 or y2 <= y1:
            return mask

        # Extract ROI
        if len(image.shape) == 3:
            roi = cv2.cvtColor(image[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
        else:
            roi = image[y1:y2, x1:x2]

        # Otsu thresholding for foreground
        _, binary = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

        # Morphological cleanup
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)

        mask[y1:y2, x1:x2] = binary

        return mask

    def _maskrcnn_segmentation(
        self, image: np.ndarray, bbox: List[int]
    ) -> Optional[np.ndarray]:
        """
        Stage 2: Mask R-CNN instance segmentation via detectron2.
        Returns None if no detection found for this bbox.
        """
        if self.detector is None:
            return None

        if len(image.shape) == 2:
            image_color = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        else:
            image_color = image

        outputs = self.detector(image_color)
        instances = outputs["instances"]

        if len(instances) == 0:
            return None

        # Find best matching instance for this bbox
        pred_boxes = instances.pred_boxes.tensor.cpu().numpy()
        pred_masks = instances.pred_masks.cpu().numpy()

        x1, y1, x2, y2 = bbox
        best_iou = 0
        best_mask = None

        for i in range(len(pred_boxes)):
            iou = self._compute_iou(bbox, pred_boxes[i].tolist())
            if iou > best_iou:
                best_iou = iou
                best_mask = pred_masks[i].astype(np.uint8) * 255

        return best_mask

    def _grabcut_refinement(
        self,
        image: np.ndarray,
        bbox: List[int],
        initial_mask: np.ndarray,
    ) -> np.ndarray:
        """
        Stage 3: GrabCut refinement using initial mask.
        Refines the segmentation boundary using graph-cut optimization.
        """
        if len(image.shape) == 2:
            image_color = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        else:
            image_color = image.copy()

        h, w = image_color.shape[:2]
        x1, y1, x2, y2 = [int(c) for c in bbox]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)

        if x2 - x1 < 5 or y2 - y1 < 5:
            return initial_mask

        # Convert initial mask to GrabCut format
        gc_mask = np.zeros((h, w), dtype=np.uint8)
        gc_mask[:] = cv2.GC_BGD  # Background

        # Set bbox region as probable foreground
        gc_mask[y1:y2, x1:x2] = cv2.GC_PR_FGD

        # Use initial mask to set definite foreground
        if initial_mask is not None:
            fg_pixels = initial_mask > 127
            gc_mask[fg_pixels] = cv2.GC_FGD

        # GrabCut models
        bgd_model = np.zeros((1, 65), dtype=np.float64)
        fgd_model = np.zeros((1, 65), dtype=np.float64)

        rect = (x1, y1, x2 - x1, y2 - y1)

        try:
            cv2.grabCut(
                image_color,
                gc_mask,
                rect,
                bgd_model,
                fgd_model,
                self.grabcut_iterations,
                cv2.GC_INIT_WITH_MASK if initial_mask is not None else cv2.GC_INIT_WITH_RECT,
            )
        except cv2.error:
            # GrabCut can fail on very small regions
            return initial_mask

        # Extract foreground
        result = np.where(
            (gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD),
            255, 0
        ).astype(np.uint8)

        return result

    @staticmethod
    def _compute_iou(box1: List[float], box2: List[float]) -> float:
        """Compute IoU between two [x1, y1, x2, y2] boxes."""
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])

        if x2 <= x1 or y2 <= y1:
            return 0.0

        intersection = (x2 - x1) * (y2 - y1)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - intersection

        return intersection / max(union, 1e-6)


def segment_objects(
    image: np.ndarray,
    bboxes: List[List[int]],
    use_detectron2: bool = False,
) -> List[np.ndarray]:
    """
    Convenience function: segment objects from bounding boxes.

    Args:
        image: Input image.
        bboxes: List of [x1, y1, x2, y2] bounding boxes.
        use_detectron2: Whether to try using detectron2.

    Returns:
        List of binary masks.
    """
    segmentor = SegmentationRefinement(use_detectron2=use_detectron2)
    return segmentor.segment(image, bboxes)



### Data Augmentation


In [ ]:
"""
Training augmentation pipelines using albumentations.
Separate pipelines for training and validation.
"""

import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
from typing import Optional


def get_train_augmentation(
    image_size: int = 640,
    rotation_limit: int = 30,
    scale_range: tuple = (0.8, 1.2),
    brightness_limit: float = 0.15,
    contrast_limit: float = 0.15,
    gauss_noise_var_limit: tuple = (10.0, 50.0),
    cutmix_prob: float = 0.0,
) -> A.Compose:
    """
    Get training augmentation pipeline.

    Includes geometric, photometric, and X-ray-specific augmentations.
    Compatible with bounding box annotations (YOLO/Pascal VOC format).

    Args:
        image_size: Target image size.
        rotation_limit: Max rotation degrees.
        scale_range: Min/max scale factors.
        brightness_limit: Max brightness/contrast adjustment.
        contrast_limit: Max contrast adjustment.
        gauss_noise_var_limit: Gaussian noise variance range.
        cutmix_prob: Probability of CutOut augmentation (simulates occlusion).

    Returns:
        albumentations Compose pipeline.
    """
    transforms = [
        # Geometric augmentations
        A.HorizontalFlip(p=0.5),
        A.Rotate(
            limit=rotation_limit,
            border_mode=0,  # cv2.BORDER_CONSTANT
            value=0,
            p=0.5,
        ),
        A.RandomScale(
            scale_limit=(scale_range[0] - 1.0, scale_range[1] - 1.0),
            p=0.3,
        ),
        A.ElasticTransform(
            alpha=50,
            sigma=5,
            p=0.1,
        ),

        # Photometric augmentations
        A.RandomBrightnessContrast(
            brightness_limit=brightness_limit,
            contrast_limit=contrast_limit,
            p=0.5,
        ),
        A.GaussNoise(
            var_limit=gauss_noise_var_limit,
            p=0.3,
        ),
        A.RandomGamma(
            gamma_limit=(80, 120),
            p=0.3,
        ),

        # X-ray specific: simulate partial occlusion
        A.CoarseDropout(
            max_holes=3,
            max_height=int(image_size * 0.15),
            max_width=int(image_size * 0.15),
            fill_value=128,
            p=cutmix_prob,
        ),

        # Resize to target
        A.Resize(image_size, image_size),
    ]

    return A.Compose(
        transforms,
        bbox_params=A.BboxParams(
            format="pascal_voc",  # [x1, y1, x2, y2]
            label_fields=["class_labels"],
            min_visibility=0.3,
        ),
    )


def get_val_augmentation(image_size: int = 640) -> A.Compose:
    """
    Get validation/test augmentation pipeline.
    Only resizing, no augmentation.
    """
    return A.Compose(
        [A.Resize(image_size, image_size)],
        bbox_params=A.BboxParams(
            format="pascal_voc",
            label_fields=["class_labels"],
            min_visibility=0.3,
        ),
    )


def get_mosaic_augmentation(
    images: list,
    bboxes_list: list,
    labels_list: list,
    image_size: int = 640,
    seed: Optional[int] = None,
) -> tuple:
    """
    Apply mosaic augmentation: combine 4 images into one.

    This is a powerful augmentation that increases diversity by
    showing multiple images in a single training sample.

    Args:
        images: List of 4 images (np.ndarray).
        bboxes_list: List of 4 bbox arrays.
        labels_list: List of 4 label arrays.
        image_size: Output image size.
        seed: Random seed.

    Returns:
        (mosaic_image, mosaic_bboxes, mosaic_labels)
    """
    rng = np.random.RandomState(seed)

    if len(images) < 4:
        # Repeat to fill 4
        while len(images) < 4:
            images.append(images[0])
            bboxes_list.append(bboxes_list[0])
            labels_list.append(labels_list[0])

    # Random center point
    cx = rng.randint(image_size // 4, 3 * image_size // 4)
    cy = rng.randint(image_size // 4, 3 * image_size // 4)

    mosaic = np.full((image_size, image_size, images[0].shape[2] if len(images[0].shape) == 3 else 1),
                     114, dtype=np.uint8)
    all_bboxes = []
    all_labels = []

    # Place 4 images in quadrants
    placements = [
        (0, 0, cx, cy),            # top-left
        (cx, 0, image_size, cy),   # top-right
        (0, cy, cx, image_size),   # bottom-left
        (cx, cy, image_size, image_size),  # bottom-right
    ]

    for i, (x1, y1, x2, y2) in enumerate(placements):
        img = images[i]
        bboxes = bboxes_list[i]
        labels = labels_list[i]

        rh, rw = y2 - y1, x2 - x1
        h, w = img.shape[:2]

        # Scale image to fit region
        scale = min(rw / w, rh / h)
        new_w, new_h = int(w * scale), int(h * scale)
        img_resized = cv2.resize(img, (new_w, new_h)) if new_w > 0 and new_h > 0 else img

        # Place in mosaic
        paste_x = x1
        paste_y = y1
        ph = min(new_h, rh)
        pw = min(new_w, rw)

        if len(img_resized.shape) == 2:
            img_resized = img_resized[:, :, np.newaxis]
        mosaic[paste_y:paste_y + ph, paste_x:paste_x + pw] = img_resized[:ph, :pw]

        # Adjust bboxes
        if len(bboxes) > 0:
            adjusted = bboxes.copy().astype(float)
            adjusted[:, [0, 2]] = adjusted[:, [0, 2]] * scale + paste_x
            adjusted[:, [1, 3]] = adjusted[:, [1, 3]] * scale + paste_y

            # Clip to mosaic bounds
            adjusted[:, [0, 2]] = np.clip(adjusted[:, [0, 2]], x1, x2)
            adjusted[:, [1, 3]] = np.clip(adjusted[:, [1, 3]], y1, y2)

            # Filter out boxes that became too small
            widths = adjusted[:, 2] - adjusted[:, 0]
            heights = adjusted[:, 3] - adjusted[:, 1]
            valid = (widths > 5) & (heights > 5)

            all_bboxes.append(adjusted[valid])
            all_labels.append(labels[valid] if isinstance(labels, np.ndarray) else np.array(labels)[valid])

    if all_bboxes:
        all_bboxes = np.concatenate(all_bboxes, axis=0)
        all_labels = np.concatenate(all_labels, axis=0)
    else:
        all_bboxes = np.zeros((0, 4))
        all_labels = np.array([])

    return mosaic, all_bboxes, all_labels


import cv2  # needed for mosaic



### Property Annotator


In [ ]:
"""
Semi-automated Property Annotator for X-Ray images.

Computes the 11-dimensional property vector for each detected object
using OpenCV and NumPy operations on the object's mask and bounding box.

Property Schema (10 + 1 occlusion):
  0. Edge Sharpness Score        (float, 0-1)
  1. Length-to-Width Ratio       (float, 0-10+)
  2. Symmetry Score              (float, 0-1)
  3. Curvature Index             (float, 0-1)
  4. Approximate Volume          (float, 0-1, normalized)
  5. Material Category           (int, 0-3: organic/metallic/mixed/opaque)
  6. Avg Absorption Intensity    (float, 0-1)
  7. Material Homogeneity        (float, 0-1)
  8. Density Level               (float, 0-1)
  9. Sharp Edge Count            (int, 0-20)
 10. Occlusion Score             (float, 0-1)
"""

import os
import cv2
import numpy as np
from typing import List, Dict, Tuple, Optional


# Material category mapping
MATERIAL_ORGANIC = 0
MATERIAL_METALLIC = 1
MATERIAL_MIXED = 2
MATERIAL_OPAQUE = 3

MATERIAL_NAMES = ["organic", "metallic", "mixed", "opaque"]

PROPERTY_NAMES = [
    "edge_sharpness",
    "length_to_width_ratio",
    "symmetry_score",
    "curvature_index",
    "approximate_volume",
    "material_category",
    "avg_absorption_intensity",
    "material_homogeneity",
    "density_level",
    "sharp_edge_count",
    "occlusion_score",
]


class PropertyAnnotator:
    """
    Computes 11-dimensional property vectors from X-ray images.

    Usage:
        annotator = PropertyAnnotator()
        img_gray = cv2.imread("xray.jpg", cv2.IMREAD_GRAYSCALE)
        mask = ...  # Binary mask of the object
        bbox = [x1, y1, x2, y2]
        properties = annotator.compute_properties(img_gray, mask, bbox)
    """

    def __init__(
        self,
        canny_low: int = 50,
        canny_high: int = 150,
        corner_max_corners: int = 100,
        corner_quality: float = 0.01,
        corner_min_distance: int = 10,
    ):
        self.canny_low = canny_low
        self.canny_high = canny_high
        self.corner_max_corners = corner_max_corners
        self.corner_quality = corner_quality
        self.corner_min_distance = corner_min_distance

    def compute_properties(
        self,
        image: np.ndarray,
        mask: np.ndarray,
        bbox: List[int],
        all_bboxes: Optional[List[List[int]]] = None,
        current_idx: int = 0,
    ) -> Dict[str, float]:
        """
        Compute the full 11-dimensional property vector for an object.

        Args:
            image: Grayscale X-ray image (H, W), uint8.
            mask: Binary mask for this object (H, W), uint8 (0 or 255).
            bbox: Bounding box [x1, y1, x2, y2].
            all_bboxes: All bounding boxes in the image (for occlusion computation).
            current_idx: Index of the current bbox in all_bboxes.

        Returns:
            Dictionary mapping property names to their values.
        """
        x1, y1, x2, y2 = bbox
        x1, y1 = max(0, int(x1)), max(0, int(y1))
        x2, y2 = min(image.shape[1], int(x2)), min(image.shape[0], int(y2))

        # Extract object region
        obj_img = image[y1:y2, x1:x2]
        obj_mask = mask[y1:y2, x1:x2] if mask is not None else np.ones_like(obj_img)

        # Ensure mask is binary
        if obj_mask.max() > 1:
            obj_mask = (obj_mask > 127).astype(np.uint8)
        else:
            obj_mask = obj_mask.astype(np.uint8)

        # Compute each property
        props = {}
        props["edge_sharpness"] = self._edge_sharpness(obj_img, obj_mask)
        props["length_to_width_ratio"] = self._length_to_width_ratio(x1, y1, x2, y2)
        props["symmetry_score"] = self._symmetry_score(obj_img, obj_mask)
        props["curvature_index"] = self._curvature_index(obj_mask)
        props["approximate_volume"] = self._approximate_volume(obj_img, obj_mask)
        props["material_category"] = self._material_category(obj_img, obj_mask)
        props["avg_absorption_intensity"] = self._avg_absorption(obj_img, obj_mask)
        props["material_homogeneity"] = self._material_homogeneity(obj_img, obj_mask)
        props["density_level"] = self._density_level(obj_img, obj_mask)
        props["sharp_edge_count"] = self._sharp_edge_count(obj_img, obj_mask)
        props["occlusion_score"] = self._occlusion_score(
            bbox, all_bboxes, current_idx
        )

        return props

    def compute_properties_vector(
        self,
        image: np.ndarray,
        mask: np.ndarray,
        bbox: List[int],
        all_bboxes: Optional[List[List[int]]] = None,
        current_idx: int = 0,
    ) -> np.ndarray:
        """Compute property vector as a numpy array (11,)."""
        props = self.compute_properties(image, mask, bbox, all_bboxes, current_idx)
        return np.array([props[name] for name in PROPERTY_NAMES], dtype=np.float32)

    # ----------------------------------------------------------------
    # Individual property computation methods
    # ----------------------------------------------------------------

    def _edge_sharpness(self, obj_img: np.ndarray, obj_mask: np.ndarray) -> float:
        """Canny edge detection density within the object mask."""
        if obj_img.size == 0 or obj_mask.sum() == 0:
            return 0.0
        edges = cv2.Canny(obj_img, self.canny_low, self.canny_high)
        masked_edges = edges * obj_mask
        edge_pixels = np.count_nonzero(masked_edges)
        mask_pixels = np.count_nonzero(obj_mask)
        return float(np.clip(edge_pixels / max(mask_pixels, 1), 0.0, 1.0))

    def _length_to_width_ratio(self, x1: int, y1: int, x2: int, y2: int) -> float:
        """Bounding box length-to-width ratio."""
        w = max(x2 - x1, 1)
        h = max(y2 - y1, 1)
        ratio = max(w, h) / min(w, h)
        return float(min(ratio, 10.0))

    def _symmetry_score(self, obj_img: np.ndarray, obj_mask: np.ndarray) -> float:
        """Symmetry via horizontal pixel distribution comparison."""
        if obj_img.size == 0 or obj_mask.sum() == 0:
            return 0.0

        masked = obj_img.astype(float) * obj_mask
        h, w = masked.shape

        # Horizontal symmetry
        if w < 2:
            return 1.0
        left = masked[:, : w // 2]
        right = np.flip(masked[:, (w - w // 2):], axis=1)

        # Make same size
        min_w = min(left.shape[1], right.shape[1])
        left = left[:, :min_w]
        right = right[:, :min_w]

        if left.size == 0:
            return 0.0

        diff = np.abs(left - right).mean()
        max_val = max(masked.max(), 1.0)
        symmetry = 1.0 - (diff / max_val)
        return float(np.clip(symmetry, 0.0, 1.0))

    def _curvature_index(self, obj_mask: np.ndarray) -> float:
        """Contour curvature analysis — ratio of contour length to convex hull length."""
        if obj_mask.size == 0 or obj_mask.sum() == 0:
            return 0.0

        mask_uint8 = (obj_mask * 255).astype(np.uint8) if obj_mask.max() <= 1 else obj_mask
        contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            return 0.0

        largest = max(contours, key=cv2.contourArea)
        perimeter = cv2.arcLength(largest, True)
        hull = cv2.convexHull(largest)
        hull_perimeter = cv2.arcLength(hull, True)

        if hull_perimeter == 0:
            return 0.0

        # Curvature index: how much the contour deviates from its convex hull
        curvature = 1.0 - (hull_perimeter / max(perimeter, 1.0))
        return float(np.clip(curvature, 0.0, 1.0))

    def _approximate_volume(self, obj_img: np.ndarray, obj_mask: np.ndarray) -> float:
        """Pixel area × opacity-estimated depth, normalized."""
        if obj_img.size == 0 or obj_mask.sum() == 0:
            return 0.0

        masked = obj_img.astype(float) * obj_mask
        area = np.count_nonzero(obj_mask)
        avg_intensity = masked.sum() / max(area, 1)

        # Approximate depth from intensity (darker = denser = thicker)
        # Invert: higher intensity in X-ray = less absorption = thinner
        depth_estimate = 1.0 - (avg_intensity / 255.0)

        # Normalize volume: area * depth / max_possible
        max_area = obj_img.shape[0] * obj_img.shape[1]
        volume = (area * depth_estimate) / max(max_area, 1)
        return float(np.clip(volume, 0.0, 1.0))

    def _material_category(self, obj_img: np.ndarray, obj_mask: np.ndarray) -> int:
        """
        Estimate material category from intensity distribution.

        For single-energy images, we use intensity thresholds as a proxy:
        - Very dark (high absorption): metallic (1)
        - Light (low absorption): organic (0)
        - Mixed range: mixed (2)
        - Uniform very dark: opaque (3)
        """
        if obj_img.size == 0 or obj_mask.sum() == 0:
            return MATERIAL_MIXED

        masked_pixels = obj_img[obj_mask > 0].astype(float)
        if len(masked_pixels) == 0:
            return MATERIAL_MIXED

        mean_val = masked_pixels.mean() / 255.0
        std_val = masked_pixels.std() / 255.0

        if mean_val < 0.2 and std_val < 0.1:
            return MATERIAL_OPAQUE
        elif mean_val < 0.4:
            return MATERIAL_METALLIC
        elif mean_val > 0.6:
            return MATERIAL_ORGANIC
        else:
            return MATERIAL_MIXED

    def _avg_absorption(self, obj_img: np.ndarray, obj_mask: np.ndarray) -> float:
        """Mean pixel value in grayscale channel, normalized to [0, 1]."""
        if obj_img.size == 0 or obj_mask.sum() == 0:
            return 0.0
        masked_pixels = obj_img[obj_mask > 0].astype(float)
        if len(masked_pixels) == 0:
            return 0.0
        return float(masked_pixels.mean() / 255.0)

    def _material_homogeneity(self, obj_img: np.ndarray, obj_mask: np.ndarray) -> float:
        """Standard deviation of absorption across object, inverted to [0, 1]."""
        if obj_img.size == 0 or obj_mask.sum() == 0:
            return 0.0
        masked_pixels = obj_img[obj_mask > 0].astype(float)
        if len(masked_pixels) < 2:
            return 1.0
        std = masked_pixels.std() / 255.0
        # Invert: high homogeneity = low std = score close to 1
        return float(np.clip(1.0 - std * 2, 0.0, 1.0))

    def _density_level(self, obj_img: np.ndarray, obj_mask: np.ndarray) -> float:
        """
        Density level from weighted intensity ratio.
        For single-energy, approximated as inverted mean intensity.
        Dark objects have higher density.
        """
        if obj_img.size == 0 or obj_mask.sum() == 0:
            return 0.0
        masked_pixels = obj_img[obj_mask > 0].astype(float)
        if len(masked_pixels) == 0:
            return 0.0
        # Invert: darker = higher density
        return float(1.0 - masked_pixels.mean() / 255.0)

    def _sharp_edge_count(self, obj_img: np.ndarray, obj_mask: np.ndarray) -> int:
        """Harris / Shi-Tomasi corner detection count."""
        if obj_img.size == 0 or obj_mask.sum() == 0:
            return 0

        # Shi-Tomasi corner detection
        masked_img = obj_img * obj_mask
        corners = cv2.goodFeaturesToTrack(
            masked_img,
            maxCorners=self.corner_max_corners,
            qualityLevel=self.corner_quality,
            minDistance=self.corner_min_distance,
        )

        count = 0 if corners is None else len(corners)
        return min(count, 20)

    def _occlusion_score(
        self,
        bbox: List[int],
        all_bboxes: Optional[List[List[int]]],
        current_idx: int,
    ) -> float:
        """Fraction of bounding box area overlapped by other bounding boxes."""
        if all_bboxes is None or len(all_bboxes) <= 1:
            return 0.0

        x1, y1, x2, y2 = bbox
        bbox_area = max((x2 - x1) * (y2 - y1), 1)
        total_overlap = 0.0

        for i, other_bbox in enumerate(all_bboxes):
            if i == current_idx:
                continue
            ox1, oy1, ox2, oy2 = other_bbox

            # Compute intersection
            ix1 = max(x1, ox1)
            iy1 = max(y1, oy1)
            ix2 = min(x2, ox2)
            iy2 = min(y2, oy2)

            if ix1 < ix2 and iy1 < iy2:
                intersection = float((ix2 - ix1) * (iy2 - iy1))
                total_overlap += intersection

        return float(np.clip(total_overlap / float(bbox_area), 0.0, 1.0))


def batch_annotate(
    image_paths: List[str],
    annotation_paths: List[str],
    output_csv: str,
    annotator: Optional[PropertyAnnotator] = None,
):
    """
    Batch compute property vectors for a dataset.

    Args:
        image_paths: List of image file paths.
        annotation_paths: List of annotation files (YOLO format .txt).
        output_csv: Output CSV path for the property manifest.
        annotator: PropertyAnnotator instance (created if None).
    """
    import pandas as pd
    from tqdm import tqdm

    if annotator is None:
        annotator = PropertyAnnotator()

    records = []

    for img_path, ann_path in tqdm(
        zip(image_paths, annotation_paths),
        total=len(image_paths),
        desc="Computing properties",
    ):
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            print(f"⚠ Could not read: {img_path}")
            continue

        h, w = image.shape[:2]

        # Parse YOLO annotations
        bboxes = []
        classes = []
        if os.path.exists(ann_path):
            with open(ann_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls = int(parts[0])
                        cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                        # Convert YOLO to pixel coords
                        x1 = int((cx - bw / 2) * w)
                        y1 = int((cy - bh / 2) * h)
                        x2 = int((cx + bw / 2) * w)
                        y2 = int((cy + bh / 2) * h)
                        bboxes.append([x1, y1, x2, y2])
                        classes.append(cls)

        # Create simple mask from bbox (when segmentation masks unavailable)
        for i, (bbox, cls) in enumerate(zip(bboxes, classes)):
            mask = np.zeros(image.shape[:2], dtype=np.uint8)
            bx1, by1, bx2, by2 = bbox
            mask[max(0, by1):by2, max(0, bx1):bx2] = 1

            props = annotator.compute_properties(
                image, mask, bbox, all_bboxes=bboxes, current_idx=i
            )

            record = {
                "image_path": img_path,
                "class_id": cls,
                "bbox_x1": bbox[0],
                "bbox_y1": bbox[1],
                "bbox_x2": bbox[2],
                "bbox_y2": bbox[3],
            }
            record.update(props)
            records.append(record)

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    print(f"✓ Saved {len(records)} property annotations to {output_csv}")
    return df



### Dataset Splits Generator


In [ ]:
"""
Dataset splitting utilities.
Generates train/val/test splits with class-balanced stratification.
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Tuple, Optional, List
from collections import Counter


def create_splits(
    image_dir: str,
    label_dir: str,
    output_dir: str,
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42,
    create_symlinks: bool = False,
) -> Tuple[List[str], List[str], List[str]]:
    """
    Create stratified train/val/test splits.

    Stratification is based on the class labels in annotation files.
    Images without annotations are placed in the non-threat class.

    Args:
        image_dir: Path to images directory.
        label_dir: Path to YOLO-format labels directory.
        output_dir: Directory to save split files (train.txt, val.txt, test.txt).
        train_ratio: Fraction for training set.
        val_ratio: Fraction for validation set.
        test_ratio: Fraction for test set.
        seed: Random seed.
        create_symlinks: If True, create directory structure with symlinks.

    Returns:
        Tuple of (train_files, val_files, test_files) - lists of image filenames.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, \
        f"Ratios must sum to 1.0, got {train_ratio + val_ratio + test_ratio}"

    rng = np.random.RandomState(seed)

    # Collect all images
    image_files = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ])

    if len(image_files) == 0:
        print("⚠ No images found in", image_dir)
        return [], [], []

    # Determine primary class for each image (for stratification)
    image_classes = {}
    for img_file in image_files:
        label_file = os.path.splitext(img_file)[0] + ".txt"
        label_path = os.path.join(label_dir, label_file)

        if os.path.exists(label_path):
            with open(label_path) as f:
                lines = f.readlines()
            if lines:
                # Use the first object's class as the stratification key
                first_class = int(lines[0].strip().split()[0])
                image_classes[img_file] = first_class
            else:
                image_classes[img_file] = -1  # Empty annotation = non-threat
        else:
            image_classes[img_file] = -1  # No annotation = non-threat

    # Group by class
    class_to_images = {}
    for img, cls in image_classes.items():
        class_to_images.setdefault(cls, []).append(img)

    # Stratified split
    train_files, val_files, test_files = [], [], []

    for cls, imgs in class_to_images.items():
        rng.shuffle(imgs)
        n = len(imgs)
        n_train = max(1, int(n * train_ratio))
        n_val = max(1, int(n * val_ratio)) if n > 2 else 0
        # Remaining go to test
        train_files.extend(imgs[:n_train])
        val_files.extend(imgs[n_train : n_train + n_val])
        test_files.extend(imgs[n_train + n_val :])

    # Shuffle within splits
    rng.shuffle(train_files)
    rng.shuffle(val_files)
    rng.shuffle(test_files)

    # Save split files
    os.makedirs(output_dir, exist_ok=True)
    _save_split_file(os.path.join(output_dir, "train.txt"), train_files, image_dir)
    _save_split_file(os.path.join(output_dir, "val.txt"), val_files, image_dir)
    _save_split_file(os.path.join(output_dir, "test.txt"), test_files, image_dir)

    # Print statistics
    print(f"\n📊 Dataset Split Statistics:")
    print(f"   Total images: {len(image_files)}")
    print(f"   Train: {len(train_files)} ({len(train_files)/len(image_files)*100:.1f}%)")
    print(f"   Val:   {len(val_files)} ({len(val_files)/len(image_files)*100:.1f}%)")
    print(f"   Test:  {len(test_files)} ({len(test_files)/len(image_files)*100:.1f}%)")

    # Class distribution per split
    print(f"\n   Class distribution:")
    for split_name, split_files in [("Train", train_files), ("Val", val_files), ("Test", test_files)]:
        class_counts = Counter(image_classes[f] for f in split_files)
        print(f"   {split_name}: {dict(class_counts)}")

    # Create symlink-based directory structure if requested
    if create_symlinks:
        _create_split_directories(
            image_dir, label_dir, output_dir,
            train_files, val_files, test_files,
        )

    return train_files, val_files, test_files


def _save_split_file(filepath: str, filenames: List[str], image_dir: str):
    """Save split file with full paths."""
    with open(filepath, "w") as f:
        for name in filenames:
            f.write(os.path.join(image_dir, name) + "\n")
    print(f"  ✓ Saved: {filepath} ({len(filenames)} images)")


def _create_split_directories(
    image_dir: str,
    label_dir: str,
    output_dir: str,
    train_files: List[str],
    val_files: List[str],
    test_files: List[str],
):
    """Create YOLO-style directory structure with copies."""
    import shutil

    for split_name, split_files in [("train", train_files), ("val", val_files), ("test", test_files)]:
        split_img_dir = os.path.join(output_dir, split_name, "images")
        split_label_dir = os.path.join(output_dir, split_name, "labels")
        os.makedirs(split_img_dir, exist_ok=True)
        os.makedirs(split_label_dir, exist_ok=True)

        for img_file in split_files:
            # Copy image
            src_img = os.path.join(image_dir, img_file)
            if os.path.exists(src_img):
                shutil.copy2(src_img, os.path.join(split_img_dir, img_file))

            # Copy label
            label_file = os.path.splitext(img_file)[0] + ".txt"
            src_label = os.path.join(label_dir, label_file)
            if os.path.exists(src_label):
                shutil.copy2(src_label, os.path.join(split_label_dir, label_file))

    print(f"  ✓ Created split directories in {output_dir}")


def generate_manifest_csv(
    split_dir: str,
    property_csv: Optional[str] = None,
    output_path: Optional[str] = None,
) -> pd.DataFrame:
    """
    Generate a master manifest CSV combining splits and property annotations.

    Args:
        split_dir: Directory containing train.txt, val.txt, test.txt.
        property_csv: Optional path to property annotations CSV.
        output_path: Path to save the manifest CSV.

    Returns:
        DataFrame with the complete manifest.
    """
    records = []

    for split in ["train", "val", "test"]:
        split_file = os.path.join(split_dir, f"{split}.txt")
        if not os.path.exists(split_file):
            continue
        with open(split_file) as f:
            for line in f:
                path = line.strip()
                if path:
                    records.append({"image_path": path, "split": split})

    df = pd.DataFrame(records)

    if property_csv and os.path.exists(property_csv):
        props_df = pd.read_csv(property_csv)
        df = df.merge(props_df, on="image_path", how="left")

    if output_path:
        df.to_csv(output_path, index=False)
        print(f"✓ Manifest saved: {output_path} ({len(df)} entries)")

    return df


def create_yolo_data_yaml(
    data_dir: str,
    class_names: List[str],
    output_path: str,
):
    """
    Create a data.yaml file for YOLOv8 training.

    Args:
        data_dir: Root directory with train/val/test subdirectories.
        class_names: List of class name strings.
        output_path: Path to save data.yaml.
    """
    import yaml

    data_config = {
        "path": os.path.abspath(data_dir),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "nc": len(class_names),
        "names": class_names,
    }

    with open(output_path, "w") as f:
        yaml.dump(data_config, f, default_flow_style=False)

    print(f"✓ Created YOLO data.yaml: {output_path}")



### Dataset Downloader & Organizer


In [ ]:
"""
Dataset download and organization utilities.
Handles downloading SIXray and OPIXray datasets.
"""

import os
import zipfile
import tarfile
import shutil
from pathlib import Path


# Dataset information
DATASET_INFO = {
    "sixray": {
        "description": "SIXray Dataset (Miao et al., CVPR 2019)",
        "url": "https://github.com/MeioJane/SIXray",
        "classes": ["gun", "knife", "wrench", "pliers", "scissors"],
        "num_images": "~8,929 positive + subset of negatives",
        "notes": (
            "SIXray is available via GitHub. The full dataset (1M+ images) "
            "requires requesting access. SIXray10 subset is commonly used. "
            "Download from the GitHub repository or use the Kaggle mirror."
        ),
    },
    "opixray": {
        "description": "OPIXray Dataset (Wei et al., ACM MM 2020)",
        "url": "https://github.com/OPIXray-author/OPIXray",
        "classes": ["folding_knife", "straight_knife", "scissor", "utility_knife", "multi_tool"],
        "num_images": 8885,
        "notes": (
            "OPIXray focuses on occluded prohibited items in X-ray images. "
            "Download from the GitHub repository."
        ),
    },
    "hixray": {
        "description": "HiXray Dataset (Tao et al., AAAI 2022)",
        "url": "https://github.com/DIG-Beihang/XrayDetection",
        "classes": [
            "portable_charger_1", "portable_charger_2", "mobile_phone",
            "laptop", "tablet", "cosmetic", "water", "nonmetallic_lighter",
        ],
        "num_images": 45364,
        "notes": (
            "HiXray provides real airport security check images. "
            "Request access from the authors."
        ),
    },
}


def setup_data_directories(data_root: str) -> dict:
    """
    Create the standardized data directory structure.

    Args:
        data_root: Root data directory path.

    Returns:
        Dictionary of created directory paths.
    """
    dirs = {
        "raw": os.path.join(data_root, "raw"),
        "raw_sixray": os.path.join(data_root, "raw", "sixray"),
        "raw_opixray": os.path.join(data_root, "raw", "opixray"),
        "processed": os.path.join(data_root, "processed"),
        "processed_images": os.path.join(data_root, "processed", "images"),
        "processed_labels": os.path.join(data_root, "processed", "labels"),
        "annotations": os.path.join(data_root, "annotations"),
        "annotations_properties": os.path.join(data_root, "annotations", "properties"),
        "augmented": os.path.join(data_root, "augmented"),
    }

    for path in dirs.values():
        os.makedirs(path, exist_ok=True)
        print(f"  ✓ Created: {path}")

    return dirs


def list_available_datasets():
    """Print information about available datasets."""
    print("=" * 60)
    print("Available X-Ray Security Datasets")
    print("=" * 60)
    for name, info in DATASET_INFO.items():
        print(f"\n📦 {name.upper()}")
        print(f"   {info['description']}")
        print(f"   URL: {info['url']}")
        print(f"   Classes: {', '.join(info['classes'])}")
        print(f"   Images: {info['num_images']}")
        print(f"   Note: {info['notes']}")
    print()


def download_from_kaggle(dataset_slug: str, download_dir: str):
    """
    Download a dataset from Kaggle (requires kaggle API token).

    Args:
        dataset_slug: Kaggle dataset slug (e.g., 'username/dataset-name').
        download_dir: Directory to download to.
    """
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files(dataset_slug, path=download_dir, unzip=True)
        print(f"✓ Downloaded {dataset_slug} to {download_dir}")
    except ImportError:
        print("⚠ Kaggle API not installed. Install with: pip install kaggle")
        print("  Then set up your API token: https://www.kaggle.com/docs/api")
    except Exception as e:
        print(f"⚠ Kaggle download failed: {e}")
        print("  Try manual download from: https://www.kaggle.com/datasets/")


def extract_archive(archive_path: str, extract_dir: str):
    """Extract zip or tar archives."""
    if archive_path.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(extract_dir)
    elif archive_path.endswith((".tar.gz", ".tgz")):
        with tarfile.open(archive_path, "r:gz") as tf:
            tf.extractall(extract_dir)
    elif archive_path.endswith(".tar"):
        with tarfile.open(archive_path, "r:") as tf:
            tf.extractall(extract_dir)
    print(f"✓ Extracted to {extract_dir}")


def organize_sixray(raw_dir: str, output_dir: str, subset: str = "SIXray10"):
    """
    Organize SIXray dataset into a standardized format.

    Expected raw structure:
        raw_dir/
        ├── SIXray10/  (or SIXray100)
        │   ├── positive/
        │   │   ├── P00001.jpg
        │   │   └── ...
        │   ├── negative/
        │   │   ├── N00001.jpg
        │   │   └── ...
        │   └── annotation/
        │       ├── gun.txt (or xmls)
        │       └── ...

    Standardized output:
        output_dir/
        ├── images/
        │   ├── img_0001.jpg
        │   └── ...
        └── labels/
            ├── img_0001.txt  (YOLO format: class x_center y_center width height)
            └── ...
    """
    img_dir = os.path.join(output_dir, "images")
    label_dir = os.path.join(output_dir, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(label_dir, exist_ok=True)

    source_dir = os.path.join(raw_dir, subset)
    if not os.path.exists(source_dir):
        print(f"⚠ Source directory not found: {source_dir}")
        print(f"  Please download SIXray and place it in: {raw_dir}")
        return

    # Process positive images
    pos_dir = os.path.join(source_dir, "positive")
    if os.path.exists(pos_dir):
        for img_name in sorted(os.listdir(pos_dir)):
            if img_name.lower().endswith((".jpg", ".png", ".jpeg")):
                src = os.path.join(pos_dir, img_name)
                dst = os.path.join(img_dir, img_name)
                shutil.copy2(src, dst)

    # Process negative images
    neg_dir = os.path.join(source_dir, "negative")
    if os.path.exists(neg_dir):
        for img_name in sorted(os.listdir(neg_dir)):
            if img_name.lower().endswith((".jpg", ".png", ".jpeg")):
                src = os.path.join(neg_dir, img_name)
                dst = os.path.join(img_dir, img_name)
                shutil.copy2(src, dst)

    total = len(os.listdir(img_dir))
    print(f"✓ Organized {total} images from SIXray {subset}")


def organize_opixray(raw_dir: str, output_dir: str):
    """
    Organize OPIXray dataset into standardized format.

    OPIXray typically comes with:
        raw_dir/
        ├── train/
        │   ├── image/
        │   └── label/
        └── test/
            ├── image/
            └── label/
    """
    img_dir = os.path.join(output_dir, "images")
    label_dir = os.path.join(output_dir, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(label_dir, exist_ok=True)

    for split in ["train", "test"]:
        split_img_dir = os.path.join(raw_dir, split, "image")
        split_label_dir = os.path.join(raw_dir, split, "label")

        if os.path.exists(split_img_dir):
            for img_name in sorted(os.listdir(split_img_dir)):
                if img_name.lower().endswith((".jpg", ".png", ".jpeg")):
                    src = os.path.join(split_img_dir, img_name)
                    dst = os.path.join(img_dir, f"{split}_{img_name}")
                    shutil.copy2(src, dst)

        if os.path.exists(split_label_dir):
            for label_name in sorted(os.listdir(split_label_dir)):
                if label_name.endswith(".txt"):
                    src = os.path.join(split_label_dir, label_name)
                    dst = os.path.join(label_dir, f"{split}_{label_name}")
                    shutil.copy2(src, dst)

    total = len(os.listdir(img_dir))
    print(f"✓ Organized {total} images from OPIXray")


def validate_dataset(data_dir: str) -> dict:
    """
    Validate a dataset directory and return statistics.

    Args:
        data_dir: Path to organized dataset directory with images/ and labels/.

    Returns:
        Dictionary with dataset statistics.
    """
    img_dir = os.path.join(data_dir, "images")
    label_dir = os.path.join(data_dir, "labels")

    stats = {
        "total_images": 0,
        "total_labels": 0,
        "images_without_labels": [],
        "labels_without_images": [],
        "image_extensions": {},
    }

    if os.path.exists(img_dir):
        images = set()
        for f in os.listdir(img_dir):
            ext = os.path.splitext(f)[1].lower()
            stats["image_extensions"][ext] = stats["image_extensions"].get(ext, 0) + 1
            images.add(os.path.splitext(f)[0])
        stats["total_images"] = len(images)

    if os.path.exists(label_dir):
        labels = set()
        for f in os.listdir(label_dir):
            if f.endswith(".txt"):
                labels.add(os.path.splitext(f)[0])
        stats["total_labels"] = len(labels)

        if os.path.exists(img_dir):
            stats["images_without_labels"] = list(images - labels)[:10]
            stats["labels_without_images"] = list(labels - images)[:10]

    print(f"\n📊 Dataset Validation: {data_dir}")
    print(f"   Images: {stats['total_images']}")
    print(f"   Labels: {stats['total_labels']}")
    print(f"   Extensions: {stats['image_extensions']}")
    if stats["images_without_labels"]:
        print(f"   ⚠ {len(stats['images_without_labels'])} images without labels (showing first 10)")
    if stats["labels_without_images"]:
        print(f"   ⚠ {len(stats['labels_without_images'])} labels without images (showing first 10)")

    return stats


# ============================================================
# Colab-specific helper
# ============================================================

def setup_colab_environment(drive_path: str = "/content/drive/MyDrive/xray_detection"):
    """
    Set up the Google Colab environment:
    1. Mount Google Drive
    2. Create directory structure on Drive
    3. Return resolved paths

    Usage in Colab:
        from src.dataset.download import setup_colab_environment
        paths = setup_colab_environment()
    """
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("✓ Google Drive mounted")
    except ImportError:
        print("⚠ Not running in Google Colab")

    dirs = setup_data_directories(os.path.join(drive_path, "data"))
    os.makedirs(os.path.join(drive_path, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(drive_path, "results"), exist_ok=True)

    return {
        "root": drive_path,
        "data": os.path.join(drive_path, "data"),
        "checkpoints": os.path.join(drive_path, "checkpoints"),
        "results": os.path.join(drive_path, "results"),
        **dirs,
    }



### PyTorch X-Ray Dataset


In [ ]:
"""
PyTorch Dataset class for X-Ray luggage scanner images.
Supports loading images with YOLO-format annotations and property vectors.
"""

import os
import cv2
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from typing import Optional, Callable, Dict, List, Tuple


class XRayDataset(Dataset):
    """
    PyTorch Dataset for X-Ray security scanner images.

    Loads images with bounding box annotations (YOLO format) and
    optional property vector labels.

    Args:
        image_dir: Path to directory containing images.
        label_dir: Path to directory containing YOLO-format .txt labels.
        property_csv: Path to CSV with property vectors (from PropertyAnnotator).
        split_file: Path to .txt file listing image paths for this split.
        transform: Optional transform function (e.g., albumentations).
        image_size: Target image size (square).
        num_properties: Number of property dimensions (default 11).
        return_masks: Whether to generate bounding box masks.
    """

    def __init__(
        self,
        image_dir: str,
        label_dir: Optional[str] = None,
        property_csv: Optional[str] = None,
        split_file: Optional[str] = None,
        transform: Optional[Callable] = None,
        image_size: int = 640,
        num_properties: int = 11,
        return_masks: bool = False,
    ):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_size = image_size
        self.num_properties = num_properties
        self.return_masks = return_masks

        # Load image file list
        if split_file and os.path.exists(split_file):
            with open(split_file) as f:
                self.image_paths = [line.strip() for line in f if line.strip()]
        else:
            self.image_paths = sorted([
                os.path.join(image_dir, f)
                for f in os.listdir(image_dir)
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
            ])

        # Load property annotations if available
        self.property_data = None
        if property_csv and os.path.exists(property_csv):
            self.property_data = pd.read_csv(property_csv)
            print(f"  ✓ Loaded {len(self.property_data)} property annotations")

        print(f"  ✓ XRayDataset initialized with {len(self.image_paths)} images")

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        """
        Returns a dictionary with:
            - 'image': Tensor (C, H, W)
            - 'boxes': Tensor (N, 4) in xyxy format
            - 'labels': Tensor (N,) class IDs
            - 'properties': Tensor (N, num_properties) if property data available
            - 'image_path': str
        """
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path)

        if image is None:
            # Return a blank sample if image can't be loaded
            return self._blank_sample(img_path)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = image.shape[:2]

        # Load YOLO-format labels
        boxes, labels = self._load_labels(img_path, orig_h, orig_w)

        # Load property vectors
        properties = self._load_properties(img_path, len(boxes))

        # Apply augmentation/transform
        if self.transform is not None:
            transformed = self.transform(
                image=image,
                bboxes=boxes if len(boxes) > 0 else [],
                class_labels=labels if len(labels) > 0 else [],
            )
            image = transformed["image"]
            if len(boxes) > 0:
                boxes = np.array(transformed["bboxes"])
                labels = np.array(transformed["class_labels"])

        # Resize to target size (letterbox)
        image, boxes = self._letterbox_resize(image, boxes, self.image_size)

        # Convert to tensor
        if isinstance(image, np.ndarray):
            image = torch.from_numpy(image).float()
            if image.dim() == 3 and image.shape[-1] in [1, 3, 4]:
                image = image.permute(2, 0, 1)  # HWC -> CHW
            image = image / 255.0  # Normalize to [0, 1]

        result = {
            "image": image,
            "boxes": torch.tensor(boxes, dtype=torch.float32) if len(boxes) > 0 else torch.zeros((0, 4)),
            "labels": torch.tensor(labels, dtype=torch.long) if len(labels) > 0 else torch.zeros(0, dtype=torch.long),
            "properties": torch.tensor(properties, dtype=torch.float32),
            "image_path": img_path,
        }

        if self.return_masks:
            result["masks"] = self._generate_masks(boxes, self.image_size)

        return result

    def _load_labels(
        self, img_path: str, img_h: int, img_w: int
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Load YOLO-format bounding box labels."""
        # Determine label file path
        img_basename = os.path.splitext(os.path.basename(img_path))[0]

        label_path = None
        if self.label_dir:
            label_path = os.path.join(self.label_dir, f"{img_basename}.txt")
        else:
            # Try same directory as image
            label_path = os.path.join(os.path.dirname(img_path), f"{img_basename}.txt")

        boxes = []
        labels = []

        if label_path and os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls = int(parts[0])
                        cx, cy, bw, bh = [float(x) for x in parts[1:5]]

                        # Convert from YOLO normalized to pixel xyxy
                        x1 = (cx - bw / 2) * img_w
                        y1 = (cy - bh / 2) * img_h
                        x2 = (cx + bw / 2) * img_w
                        y2 = (cy + bh / 2) * img_h

                        boxes.append([x1, y1, x2, y2])
                        labels.append(cls)

        return np.array(boxes, dtype=np.float32).reshape(-1, 4), np.array(labels, dtype=np.int64)

    def _load_properties(
        self, img_path: str, num_objects: int
    ) -> np.ndarray:
        """Load property vectors for objects in this image."""
        if self.property_data is None or num_objects == 0:
            return np.zeros((max(num_objects, 0), self.num_properties), dtype=np.float32)

        # Match by image path
        img_props = self.property_data[
            self.property_data["image_path"] == img_path
        ]

        if len(img_props) == 0:
            return np.zeros((num_objects, self.num_properties), dtype=np.float32)

        # Extract property columns
        prop_columns = [
            "edge_sharpness", "length_to_width_ratio", "symmetry_score",
            "curvature_index", "approximate_volume", "material_category",
            "avg_absorption_intensity", "material_homogeneity",
            "density_level", "sharp_edge_count", "occlusion_score",
        ]

        available_cols = [c for c in prop_columns if c in img_props.columns]
        if available_cols:
            properties = img_props[available_cols].values.astype(np.float32)
        else:
            properties = np.zeros((len(img_props), self.num_properties), dtype=np.float32)

        # Pad or trim to match num_objects
        if len(properties) < num_objects:
            pad = np.zeros((num_objects - len(properties), properties.shape[1]), dtype=np.float32)
            properties = np.vstack([properties, pad])
        elif len(properties) > num_objects:
            properties = properties[:num_objects]

        return properties

    def _letterbox_resize(
        self,
        image: np.ndarray,
        boxes: np.ndarray,
        target_size: int,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Resize image maintaining aspect ratio with padding."""
        h, w = image.shape[:2]
        scale = min(target_size / h, target_size / w)
        new_h, new_w = int(h * scale), int(w * scale)

        image = cv2.resize(image, (new_w, new_h))

        # Pad to target size
        pad_h = target_size - new_h
        pad_w = target_size - new_w
        top = pad_h // 2
        left = pad_w // 2

        image = cv2.copyMakeBorder(
            image, top, pad_h - top, left, pad_w - left,
            cv2.BORDER_CONSTANT, value=(114, 114, 114),
        )

        # Adjust bounding boxes
        if len(boxes) > 0:
            boxes = boxes * scale
            boxes[:, [0, 2]] += left
            boxes[:, [1, 3]] += top

        return image, boxes

    def _generate_masks(
        self, boxes: np.ndarray, img_size: int
    ) -> torch.Tensor:
        """Generate binary masks from bounding boxes."""
        num_objects = len(boxes)
        masks = torch.zeros((max(num_objects, 0), img_size, img_size))

        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = [int(c) for c in box]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img_size, x2), min(img_size, y2)
            masks[i, y1:y2, x1:x2] = 1.0

        return masks

    def _blank_sample(self, img_path: str) -> Dict[str, torch.Tensor]:
        """Return a blank sample for failed image loads."""
        return {
            "image": torch.zeros((3, self.image_size, self.image_size)),
            "boxes": torch.zeros((0, 4)),
            "labels": torch.zeros(0, dtype=torch.long),
            "properties": torch.zeros((0, self.num_properties)),
            "image_path": img_path,
        }


def collate_fn(batch: List[dict]) -> dict:
    """
    Custom collate function for DataLoader.

    Since different images may have different numbers of detections,
    boxes, labels, and properties are returned as lists, not stacked tensors.
    """
    return {
        "image": torch.stack([item["image"] for item in batch]),
        "boxes": [item["boxes"] for item in batch],
        "labels": [item["labels"] for item in batch],
        "properties": [item["properties"] for item in batch],
        "image_path": [item["image_path"] for item in batch],
    }



### PropertyYOLO Architecture


In [ ]:
"""
Modified YOLOv8 architecture with Property Regression Head.

Extends Ultralytics YOLOv8 with:
  - Head 1: Standard detection head (bbox + objectness + class)
  - Head 2: Property Regression Head (3-layer MLP: 512→256→10)
  - Material Branch: 4-class material classifier (optional)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Dict, List, Tuple


class PropertyRegressionHead(nn.Module):
    """
    3-layer MLP that regresses the 10-dimensional continuous property vector
    from the same feature maps used by the detection head.

    Architecture: input_dim → 512 → BN → ReLU → 256 → BN → ReLU → 10
    """

    def __init__(
        self,
        input_dim: int = 512,
        hidden_dims: List[int] = [512, 256],
        num_properties: int = 10,
        dropout: float = 0.1,
    ):
        super().__init__()

        layers = []
        prev_dim = input_dim
        for hdim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hdim),
                nn.BatchNorm1d(hdim),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            ])
            prev_dim = hdim

        layers.append(nn.Linear(prev_dim, num_properties))

        self.mlp = nn.Sequential(*layers)

        # Sigmoid for bounded [0,1] properties, applied selectively
        self.num_properties = num_properties

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            features: (N, input_dim) pooled ROI features, or
                      (B, input_dim, H, W) feature map (will be GAP'ed).

        Returns:
            (N, num_properties) predicted property vectors.
        """
        if features.dim() == 4:
            # Global Average Pooling
            features = F.adaptive_avg_pool2d(features, 1).flatten(1)
        elif features.dim() == 3:
            features = features.mean(dim=1)

        props = self.mlp(features)

        # Apply sigmoid to properties that should be in [0, 1]
        # Properties 0-4, 6-8: float [0,1], property 5 (material): categorical,
        # property 9 (sharp_edge_count): integer
        props_bounded = torch.sigmoid(props[:, :5])
        props_material = props[:, 5:6]  # Will use cross-entropy separately
        props_bounded2 = torch.sigmoid(props[:, 6:9])
        props_count = torch.relu(props[:, 9:10])  # Non-negative integer count

        return torch.cat([props_bounded, props_material, props_bounded2, props_count], dim=1)


class MaterialClassificationBranch(nn.Module):
    """
    Shallow CNN branch for 4-class material classification
    (organic / metallic / mixed / opaque).

    Operates on the material-discriminative channel (HE-LE difference).
    """

    def __init__(
        self,
        input_channels: int = 1,
        num_classes: int = 4,
    ):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(input_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, 1, H, W) material channel (HE-LE difference).

        Returns:
            (B, num_classes) logits.
        """
        features = self.conv(x).flatten(1)
        return self.fc(features)


class PropertyYOLO(nn.Module):
    """
    Dual-head YOLOv8: detection + property extraction.

    Wraps Ultralytics YOLOv8 model and adds a PropertyRegressionHead
    that branches from the backbone feature maps.

    Usage:
        model = PropertyYOLO(model_size="yolov8m", num_classes=6)
        # In training:
        det_loss = model.train_detection(images, targets)  # Stage 1
        prop_loss = model.train_properties(images, property_targets)  # Stage 2
        # In inference:
        detections, property_vectors = model(images)
    """

    def __init__(
        self,
        model_size: str = "yolov8m",
        num_classes: int = 6,
        num_properties: int = 10,
        input_channels: int = 4,
        pretrained: bool = True,
        property_head_dims: List[int] = [512, 256],
        property_dropout: float = 0.1,
        material_branch: bool = True,
        num_materials: int = 4,
    ):
        super().__init__()

        self.num_properties = num_properties
        self.num_classes = num_classes
        self.input_channels = input_channels
        self.material_branch_enabled = material_branch

        # Initialize YOLOv8 backbone
        self._init_yolo(model_size, num_classes, pretrained, input_channels)

        # Determine backbone feature dimension
        self.backbone_dim = self._get_backbone_dim()

        # Property Regression Head
        self.property_head = PropertyRegressionHead(
            input_dim=self.backbone_dim,
            hidden_dims=property_head_dims,
            num_properties=num_properties,
            dropout=property_dropout,
        )

        # Material Classification Branch
        self.material_branch = None
        if material_branch:
            self.material_branch = MaterialClassificationBranch(
                input_channels=1,
                num_classes=num_materials,
            )

        # Stage tracking
        self.training_stage = 1  # 1 = detection, 2 = property regression

    def _init_yolo(self, model_size: str, num_classes: int, pretrained: bool, input_channels: int):
        """Initialize the Ultralytics YOLOv8 model natively with 4-channels to prevent reset bugs."""
        from ultralytics import YOLO
        import ultralytics
        from pathlib import Path
        import yaml
        
        # If input channels is 3, just use standard loading
        if input_channels == 3:
            self.yolo = YOLO(f"{model_size}.pt" if pretrained else f"{model_size}.yaml")
            return

        # For 4 channels, we dynamically create a custom yaml architecture
        # so YOLO's internal .train() method doesn't panic and download COCO
        base_yaml = f"{model_size.replace('yolov8', '')}.yaml"
        yaml_path = Path(ultralytics.__file__).parent / "cfg" / "models" / "v8" / "yolov8.yaml"
        
        custom_yaml_path = f"custom_4ch_{model_size}.yaml"
        
        # Create the custom configuration
        if yaml_path.exists():
            with open(yaml_path, "r") as f:
                d = yaml.safe_load(f)
            d["ch"] = input_channels
            d["nc"] = num_classes
            with open(custom_yaml_path, "w") as f:
                yaml.dump(d, f)
            # Initialize from custom YAML so the backbone is permanently 4 channels
            self.yolo = YOLO(custom_yaml_path)
        else:
            # Fallback if standard yaml is surprisingly missing
            self.yolo = YOLO(f"{model_size}.yaml")
            self._modify_input_channels(input_channels)

        # Load pretrained weights into the new 4-channel model
        if pretrained:
            self.yolo.load(f"{model_size}.pt")

    def _modify_input_channels(self, new_channels: int):
        """
        Fallback method: Modify the first convolutional layer inline.
        """
        model = self.yolo.model

        # Access the first conv layer
        first_conv = None
        for module in model.modules():
            if isinstance(module, nn.Conv2d):
                first_conv = module
                break

        if first_conv is not None and first_conv.in_channels != new_channels:
            old_weight = first_conv.weight.data
            new_conv = nn.Conv2d(
                new_channels,
                first_conv.out_channels,
                first_conv.kernel_size,
                stride=first_conv.stride,
                padding=first_conv.padding,
                bias=first_conv.bias is not None,
            )

            # Initialize new weights
            with torch.no_grad():
                if new_channels > first_conv.in_channels:
                    # Copy existing weights and initialize extra channels
                    new_conv.weight[:, :first_conv.in_channels] = old_weight
                    nn.init.kaiming_normal_(
                        new_conv.weight[:, first_conv.in_channels:],
                        mode="fan_out",
                    )
                else:
                    new_conv.weight = nn.Parameter(old_weight[:, :new_channels])

                if first_conv.bias is not None:
                    new_conv.bias = nn.Parameter(first_conv.bias.data.clone())

            # Replace the conv layer
            for name, module in model.named_modules():
                if module is first_conv:
                    parent_name = ".".join(name.split(".")[:-1])
                    child_name = name.split(".")[-1]
                    parent = dict(model.named_modules())[parent_name] if parent_name else model
                    setattr(parent, child_name, new_conv)
                    break

    def _get_backbone_dim(self) -> int:
        """Determine the backbone output feature dimension."""
        # YOLOv8 sizes and their channel widths at the neck output
        # This is approximate — exact value depends on model size
        try:
            # Try to get from model
            model = self.yolo.model
            for module in reversed(list(model.modules())):
                if isinstance(module, nn.Conv2d):
                    return module.out_channels
        except Exception:
            pass
        return 512  # Default fallback

    def extract_features(self, images: torch.Tensor) -> torch.Tensor:
        """
        Extract feature maps from YOLOv8 backbone.

        Args:
            images: (B, C, H, W) input tensor.

        Returns:
            (B, backbone_dim, H', W') feature maps.
        """
        model = self.yolo.model

        # Run through backbone layers to get feature maps
        x = images
        features = None

        for i, layer in enumerate(model.model):
            x = layer(x)
            # Store intermediate features
            if hasattr(layer, 'f') and layer.f == -1:
                features = x

        # Return the last backbone feature map
        if features is None:
            features = x

        return features

    def forward_properties(
        self,
        images: torch.Tensor,
        roi_features: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass for property extraction.

        Args:
            images: (B, C, H, W) preprocessed X-ray images.
            roi_features: Optional pre-extracted ROI features.

        Returns:
            Dict with 'properties' and optionally 'material_logits'.
        """
        if roi_features is None:
            roi_features = self.extract_features(images)

        result = {}

        # Property regression
        result["properties"] = self.property_head(roi_features)

        # Material classification
        if self.material_branch is not None and images.shape[1] >= 4:
            # Use the 4th channel (HE-LE) for material prediction
            material_input = images[:, 3:4, :, :]
            result["material_logits"] = self.material_branch(material_input)

        return result

    def set_training_stage(self, stage: int):
        """
        Set the training stage.

        Stage 1: Train detection only (freeze property head)
        Stage 2: Train property head only (freeze backbone)
        """
        self.training_stage = stage

        if stage == 1:
            # Freeze property head, train detection backbone
            for param in self.property_head.parameters():
                param.requires_grad = False
            if self.material_branch:
                for param in self.material_branch.parameters():
                    param.requires_grad = False
            print("⚙ Stage 1: Training detection backbone only")

        elif stage == 2:
            # Freeze backbone, train property head
            for param in self.yolo.model.parameters():
                param.requires_grad = False
            for param in self.property_head.parameters():
                param.requires_grad = True
            if self.material_branch:
                for param in self.material_branch.parameters():
                    param.requires_grad = True
            print("⚙ Stage 2: Training property head only (backbone frozen)")

    def get_trainable_params(self) -> List[torch.Tensor]:
        """Get parameters that require gradients."""
        return [p for p in self.parameters() if p.requires_grad]

    def save_checkpoint(self, path: str, epoch: int, optimizer=None, metrics=None):
        """Save model checkpoint."""
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": self.state_dict(),
            "num_properties": self.num_properties,
            "num_classes": self.num_classes,
            "input_channels": self.input_channels,
            "training_stage": self.training_stage,
        }
        if optimizer:
            checkpoint["optimizer_state_dict"] = optimizer.state_dict()
        if metrics:
            checkpoint["metrics"] = metrics

        torch.save(checkpoint, path)
        print(f"✓ Checkpoint saved: {path}")

    @classmethod
    def load_checkpoint(cls, path: str, device: str = "cpu") -> "PropertyYOLO":
        """Load model from checkpoint."""
        checkpoint = torch.load(path, map_location=device)
        model = cls(
            num_properties=checkpoint.get("num_properties", 10),
            num_classes=checkpoint.get("num_classes", 6),
            input_channels=checkpoint.get("input_channels", 4),
        )
        model.load_state_dict(checkpoint["model_state_dict"])
        model.training_stage = checkpoint.get("training_stage", 1)
        return model



### Multi-Task Loss


In [ ]:
"""
Multi-task loss function for Model 1.

Combines:
  - Detection loss (from YOLOv8)
  - Property regression loss (MSE)
  - Material classification loss (CrossEntropy)

Uses Uncertainty Weighting (Kendall et al., 2018) to automatically
balance loss contributions across tasks.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Optional


class UncertaintyWeight(nn.Module):
    """
    Learnable uncertainty parameter for a task.

    Implements Kendall et al., 2018: "Multi-Task Learning Using
    Uncertainty to Weigh Losses for Scene Geometry and Semantics"

    The loss is scaled as: L_task / (2 * sigma^2) + log(sigma)
    where sigma is a learnable parameter. This automatically
    down-weights noisy tasks and up-weights reliable ones.
    """

    def __init__(self, init_sigma: float = 1.0):
        super().__init__()
        # We learn log(sigma^2) for numerical stability
        self.log_sigma_sq = nn.Parameter(
            torch.tensor(2.0 * torch.tensor(init_sigma).log())
        )

    def forward(self, loss: torch.Tensor) -> torch.Tensor:
        """Scale loss by uncertainty weight."""
        precision = torch.exp(-self.log_sigma_sq)
        weighted_loss = precision * loss + self.log_sigma_sq
        return weighted_loss

    @property
    def sigma(self) -> float:
        """Current sigma value."""
        return (0.5 * self.log_sigma_sq).exp().item()


class FocalLoss(nn.Module):
    """
    Focal Loss for addressing class imbalance.

    FL(p) = -alpha * (1-p)^gamma * log(p)
    """

    def __init__(self, gamma: float = 2.0, alpha: float = 0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(inputs, targets, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


class MultiTaskLoss(nn.Module):
    """
    Multi-task loss for the dual-head YOLOv8 model.

    Combines detection, property regression, and material classification
    losses with automatic uncertainty weighting.

    Usage:
        criterion = MultiTaskLoss(
            num_properties=10,
            num_materials=4,
            use_uncertainty_weighting=True,
        )
        loss, loss_dict = criterion(predictions, targets)
    """

    def __init__(
        self,
        num_properties: int = 10,
        num_materials: int = 4,
        use_uncertainty_weighting: bool = True,
        use_focal_loss: bool = True,
        focal_gamma: float = 2.0,
        focal_alpha: float = 0.25,
        property_loss_type: str = "mse",  # "mse" or "smooth_l1"
    ):
        super().__init__()

        self.num_properties = num_properties
        self.num_materials = num_materials
        self.use_uncertainty = use_uncertainty_weighting
        self.property_loss_type = property_loss_type

        # Property regression loss
        if property_loss_type == "mse":
            self.property_loss_fn = nn.MSELoss()
        elif property_loss_type == "smooth_l1":
            self.property_loss_fn = nn.SmoothL1Loss()
        else:
            raise ValueError(f"Unknown property loss: {property_loss_type}")

        # Material classification loss
        if use_focal_loss:
            self.material_loss_fn = FocalLoss(gamma=focal_gamma, alpha=focal_alpha)
        else:
            self.material_loss_fn = nn.CrossEntropyLoss()

        # Uncertainty weights
        if use_uncertainty_weighting:
            self.uw_detection = UncertaintyWeight(init_sigma=1.0)
            self.uw_property = UncertaintyWeight(init_sigma=1.0)
            self.uw_material = UncertaintyWeight(init_sigma=1.0)

    def forward(
        self,
        predictions: Dict[str, torch.Tensor],
        targets: Dict[str, torch.Tensor],
        detection_loss: Optional[torch.Tensor] = None,
    ) -> tuple:
        """
        Compute combined multi-task loss.

        Args:
            predictions: Dict with keys:
                - 'properties': (N, num_properties) predicted property vectors
                - 'material_logits': (N, num_materials) material class logits (optional)
            targets: Dict with keys:
                - 'properties': (N, num_properties) ground truth property vectors
                - 'material_labels': (N,) ground truth material class indices (optional)
            detection_loss: Optional detection loss from YOLOv8 (precomputed).

        Returns:
            Tuple of (total_loss, loss_dict) where loss_dict contains individual losses.
        """
        loss_dict = {}
        total_loss = torch.tensor(0.0, device=self._get_device(predictions))

        # Detection loss (from YOLOv8, passed in)
        if detection_loss is not None:
            if self.use_uncertainty:
                det_loss = self.uw_detection(detection_loss)
            else:
                det_loss = detection_loss
            total_loss = total_loss + det_loss
            loss_dict["detection"] = detection_loss.item()
            loss_dict["detection_weighted"] = det_loss.item()

        # Property regression loss
        if "properties" in predictions and "properties" in targets:
            pred_props = predictions["properties"]
            gt_props = targets["properties"]

            if pred_props.shape[0] > 0 and gt_props.shape[0] > 0:
                # Separate continuous and categorical properties
                # Continuous: indices 0-4, 6-8, 9  |  Categorical: index 5
                continuous_idx = [0, 1, 2, 3, 4, 6, 7, 8, 9]
                cat_idx = 5

                prop_loss = self.property_loss_fn(
                    pred_props[:, continuous_idx],
                    gt_props[:, continuous_idx],
                )

                if self.use_uncertainty:
                    prop_loss = self.uw_property(prop_loss)

                total_loss = total_loss + prop_loss
                loss_dict["property_regression"] = prop_loss.item()

        # Material classification loss
        if (
            "material_logits" in predictions
            and "material_labels" in targets
        ):
            mat_logits = predictions["material_logits"]
            mat_labels = targets["material_labels"]

            if mat_logits.shape[0] > 0 and mat_labels.shape[0] > 0:
                mat_loss = self.material_loss_fn(mat_logits, mat_labels.long())

                if self.use_uncertainty:
                    mat_loss = self.uw_material(mat_loss)

                total_loss = total_loss + mat_loss
                loss_dict["material_classification"] = mat_loss.item()

        loss_dict["total"] = total_loss.item()

        # Log uncertainty weights
        if self.use_uncertainty:
            loss_dict["sigma_detection"] = self.uw_detection.sigma
            loss_dict["sigma_property"] = self.uw_property.sigma
            loss_dict["sigma_material"] = self.uw_material.sigma

        return total_loss, loss_dict

    def _get_device(self, predictions: Dict[str, torch.Tensor]) -> torch.device:
        """Get the device from predictions."""
        for v in predictions.values():
            if isinstance(v, torch.Tensor):
                return v.device
        return torch.device("cpu")



### Trainer Logic


In [ ]:
"""
Training script for Model 1 (Property Extraction).

Two-stage training:
  Stage 1 (50-100 epochs): Detection only. Backbone from COCO pretrained weights.
  Stage 2 (30-50 epochs): Freeze backbone. Train property head + material branch.

Designed for Google Colab execution.
"""

import os
import time
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from typing import Optional, Dict



class Trainer:
    """
    Two-stage trainer for the PropertyYOLO model.

    Usage:
        trainer = Trainer(config)
        # Stage 1: Detection
        trainer.train_stage1(train_loader, val_loader)
        # Stage 2: Property regression
        trainer.train_stage2(train_loader, val_loader)
    """

    def __init__(
        self,
        config: dict,
        model: Optional[PropertyYOLO] = None,
        device: str = "auto",
        checkpoint_dir: str = "checkpoints",
        use_wandb: bool = False,
    ):
        self.config = config
        self.checkpoint_dir = checkpoint_dir
        self.use_wandb = use_wandb
        self.history = {"train_loss": [], "val_loss": [], "metrics": {}}

        # Device
        if device == "auto":
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(device)
        print(f"⚙ Using device: {self.device}")

        # Model
        if model is not None:
            self.model = model.to(self.device)
        else:
            model_cfg = config.get("model1", {})
            self.model = PropertyYOLO(
                model_size=model_cfg.get("backbone", "yolov8m"),
                num_classes=len(config.get("dataset", {}).get("threat_classes", ["threat"])) + 1,
                num_properties=model_cfg.get("property_head", {}).get("num_outputs", 10),
                input_channels=model_cfg.get("input_channels", 4),
                pretrained=model_cfg.get("pretrained", True),
                property_head_dims=model_cfg.get("property_head", {}).get("hidden_dims", [512, 256]),
                material_branch=model_cfg.get("material_branch", {}).get("enabled", True),
            ).to(self.device)

        # Loss
        loss_cfg = config.get("model1", {}).get("loss", {})
        self.criterion = MultiTaskLoss(
            use_uncertainty_weighting=loss_cfg.get("uncertainty_weighting", True),
            use_focal_loss=True,
            focal_gamma=loss_cfg.get("focal_loss", {}).get("gamma", 2.0),
            focal_alpha=loss_cfg.get("focal_loss", {}).get("alpha", 0.25),
            property_loss_type=loss_cfg.get("property_loss", "mse"),
        ).to(self.device)

        self.gradient_clip = loss_cfg.get("gradient_clip_max_norm", 1.0)

        # Create checkpoint directory
        os.makedirs(checkpoint_dir, exist_ok=True)

        # Initialize W&B
        if use_wandb:
            self._init_wandb(config)

    def _init_wandb(self, config: dict):
        """Initialize Weights & Biases logging."""
        try:
            import wandb
            wandb_cfg = config.get("wandb", {})
            wandb.init(
                project=wandb_cfg.get("project", "xray-detection"),
                entity=wandb_cfg.get("entity"),
                config=config,
            )
            self.wandb = wandb
        except ImportError:
            print("⚠ wandb not installed, disabling experiment tracking")
            self.use_wandb = False

    def train_stage1(
        self,
        train_loader,
        val_loader=None,
        epochs: Optional[int] = None,
        lr: Optional[float] = None,
    ):
        """
        Stage 1: Train detection only using YOLOv8's built-in training.

        This leverages Ultralytics' optimized training pipeline for
        detection, which handles the standard YOLO losses internally.
        """
        stage1_cfg = self.config.get("model1", {}).get("stage1", {})
        epochs = epochs or stage1_cfg.get("epochs", 100)
        data_yaml = self.config.get("data_yaml_path", "data/data.yaml")

        print(f"\n{'='*60}")
        print(f"  STAGE 1: Detection Training ({epochs} epochs)")
        print(f"{'='*60}")

        self.model.set_training_stage(1)

        # Use Ultralytics training for detection
        # This is the most efficient approach as it uses their optimized
        # training loop, data loading, and augmentation
        results = self.model.yolo.train(
            data=data_yaml,
            epochs=epochs,
            imgsz=self.config.get("dataset", {}).get("image_size", 640),
            batch=stage1_cfg.get("batch_size", 16),
            lr0=stage1_cfg.get("learning_rate", 0.01),
            optimizer=stage1_cfg.get("optimizer", "SGD"),
            momentum=stage1_cfg.get("momentum", 0.937),
            weight_decay=stage1_cfg.get("weight_decay", 0.0005),
            warmup_epochs=stage1_cfg.get("warmup_epochs", 3),
            cos_lr=stage1_cfg.get("scheduler", "cosine") == "cosine",
            project=self.checkpoint_dir,
            name="stage1_detection",
            exist_ok=True,
            verbose=True,
        )

        # Save Stage 1 checkpoint
        self.model.save_checkpoint(
            os.path.join(self.checkpoint_dir, "stage1_best.pth"),
            epoch=epochs,
            metrics={"stage": 1},
        )

        print(f"✓ Stage 1 complete. Detection model saved.")
        return results

    def train_stage2(
        self,
        train_loader,
        val_loader=None,
        epochs: Optional[int] = None,
        lr: Optional[float] = None,
    ):
        """
        Stage 2: Train property regression head (backbone frozen).

        Custom training loop since Ultralytics doesn't support
        property regression natively.
        """
        stage2_cfg = self.config.get("model1", {}).get("stage2", {})
        epochs = epochs or stage2_cfg.get("epochs", 50)
        lr = lr or stage2_cfg.get("learning_rate", 0.001)

        print(f"\n{'='*60}")
        print(f"  STAGE 2: Property Regression Training ({epochs} epochs)")
        print(f"{'='*60}")

        self.model.set_training_stage(2)

        # Optimizer — only train property head parameters
        trainable_params = list(self.model.property_head.parameters())
        if self.model.material_branch is not None:
            trainable_params += list(self.model.material_branch.parameters())
        # Add uncertainty weight parameters
        trainable_params += list(self.criterion.parameters())

        optimizer = torch.optim.Adam(
            trainable_params,
            lr=lr,
            weight_decay=stage2_cfg.get("weight_decay", 0.0001),
        )

        # Cosine annealing scheduler
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs, eta_min=lr * 0.01
        )

        best_val_loss = float("inf")

        for epoch in range(1, epochs + 1):
            # Training
            train_loss = self._train_epoch_stage2(train_loader, optimizer, epoch)
            self.history["train_loss"].append(train_loss)

            # Validation
            val_loss = None
            if val_loader is not None:
                val_loss = self._validate_epoch_stage2(val_loader, epoch)
                self.history["val_loss"].append(val_loss)

            scheduler.step()

            # Logging
            lr_current = optimizer.param_groups[0]["lr"]
            log_msg = f"  Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f}"
            if val_loss is not None:
                log_msg += f" | Val Loss: {val_loss:.4f}"
            log_msg += f" | LR: {lr_current:.6f}"
            print(log_msg)

            if self.use_wandb:
                log_data = {"epoch": epoch, "train_loss": train_loss, "lr": lr_current}
                if val_loss is not None:
                    log_data["val_loss"] = val_loss
                self.wandb.log(log_data)

            # Save best model
            eval_loss = val_loss if val_loss is not None else train_loss
            if eval_loss < best_val_loss:
                best_val_loss = eval_loss
                self.model.save_checkpoint(
                    os.path.join(self.checkpoint_dir, "stage2_best.pth"),
                    epoch=epoch,
                    optimizer=optimizer,
                    metrics={"val_loss": eval_loss},
                )

            # Save periodic checkpoints
            if epoch % 10 == 0:
                self.model.save_checkpoint(
                    os.path.join(self.checkpoint_dir, f"stage2_epoch{epoch}.pth"),
                    epoch=epoch,
                )

        print(f"✓ Stage 2 complete. Best val loss: {best_val_loss:.4f}")

    def _train_epoch_stage2(self, train_loader, optimizer, epoch: int) -> float:
        """Train one epoch for property regression."""
        self.model.train()
        total_loss = 0.0
        num_batches = 0

        for batch_idx, batch in enumerate(train_loader):
            images = batch["image"].to(self.device)
            properties = batch["properties"]
            # Flatten properties for all objects in the batch
            gt_properties = torch.cat([p for p in properties if len(p) > 0], dim=0)

            if len(gt_properties) == 0:
                continue

            gt_properties = gt_properties.to(self.device)

            # Forward pass
            predictions = self.model.forward_properties(images)

            # Compute property regression loss
            targets = {"properties": gt_properties}

            # Material labels (from property index 5)
            if "material_logits" in predictions:
                material_labels = gt_properties[:, 5].long()
                targets["material_labels"] = material_labels

            loss, loss_dict = self.criterion(predictions, targets)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            nn.utils.clip_grad_norm_(
                self.model.get_trainable_params(),
                max_norm=self.gradient_clip,
            )

            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

            if batch_idx % 50 == 0:
                print(f"    Batch {batch_idx} | Loss: {loss.item():.4f}")

        return total_loss / max(num_batches, 1)

    @torch.no_grad()
    def _validate_epoch_stage2(self, val_loader, epoch: int) -> float:
        """Validate one epoch for property regression."""
        self.model.eval()
        total_loss = 0.0
        num_batches = 0

        for batch in val_loader:
            images = batch["image"].to(self.device)
            properties = batch["properties"]
            gt_properties = torch.cat([p for p in properties if len(p) > 0], dim=0)

            if len(gt_properties) == 0:
                continue

            gt_properties = gt_properties.to(self.device)

            predictions = self.model.forward_properties(images)
            targets = {"properties": gt_properties}

            if "material_logits" in predictions:
                targets["material_labels"] = gt_properties[:, 5].long()

            loss, _ = self.criterion(predictions, targets)
            total_loss += loss.item()
            num_batches += 1

        return total_loss / max(num_batches, 1)

    def get_training_history(self) -> dict:
        """Return training history for plotting."""
        return self.history


def train_model1(
    config: dict,
    train_loader,
    val_loader=None,
    checkpoint_dir: str = "checkpoints",
):
    """
    Convenience function to train Model 1 end-to-end.

    Args:
        config: Configuration dictionary.
        train_loader: Training DataLoader.
        val_loader: Validation DataLoader.
        checkpoint_dir: Directory to save checkpoints.
    """
    trainer = Trainer(
        config=config,
        checkpoint_dir=checkpoint_dir,
        use_wandb=config.get("wandb", {}).get("enabled", False),
    )

    # Stage 1: Detection
    trainer.train_stage1(train_loader, val_loader)

    # Stage 2: Property regression
    trainer.train_stage2(train_loader, val_loader)

    return trainer



### Visualization Tools


In [ ]:
"""
Visualization utilities for X-Ray Object Detection project.
Provides plotting functions for detections, training curves, and analysis.
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from typing import List, Dict, Optional, Tuple


def visualize_detections(
    image: np.ndarray,
    boxes: List[List[float]],
    labels: List[str],
    property_vectors: Optional[List[Dict[str, float]]] = None,
    scores: Optional[List[float]] = None,
    save_path: Optional[str] = None,
    figsize: Tuple[int, int] = (12, 8),
) -> plt.Figure:
    """
    Visualize detected objects on an X-ray image with bounding boxes and property vectors.

    Args:
        image: Input image (H, W) or (H, W, C).
        boxes: List of [x1, y1, x2, y2] bounding boxes.
        labels: List of class label strings.
        property_vectors: Optional list of property dicts for each detection.
        scores: Optional confidence scores for each detection.
        save_path: Optional path to save the figure.
        figsize: Figure size.

    Returns:
        matplotlib Figure object.
    """
    fig, ax = plt.subplots(1, 1, figsize=figsize)

    # Display image
    if len(image.shape) == 2:
        ax.imshow(image, cmap="gray")
    elif image.shape[2] == 4:
        # 4-channel: show first 3 as RGB
        ax.imshow(image[:, :, :3])
    else:
        ax.imshow(image)

    # Color map for classes
    colors = plt.cm.Set1(np.linspace(0, 1, max(len(set(labels)), 1)))
    unique_labels = list(set(labels))
    color_map = {label: colors[i] for i, label in enumerate(unique_labels)}

    for i, (box, label) in enumerate(zip(boxes, labels)):
        x1, y1, x2, y2 = box
        w, h = x2 - x1, y2 - y1
        color = color_map[label]

        # Draw bounding box
        rect = patches.Rectangle(
            (x1, y1), w, h, linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)

        # Build label text
        text = label
        if scores is not None and i < len(scores):
            text += f" ({scores[i]:.2f})"

        ax.text(
            x1, y1 - 5, text,
            color="white", fontsize=9, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.8),
        )

        # Show property vector if available
        if property_vectors is not None and i < len(property_vectors):
            props = property_vectors[i]
            prop_text = "\n".join([f"{k}: {v:.3f}" for k, v in props.items()])
            ax.text(
                x2 + 5, y1, prop_text,
                color="white", fontsize=7,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.7),
                verticalalignment="top",
            )

    ax.set_title("X-Ray Detection Results", fontsize=14, fontweight="bold")
    ax.axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig


def plot_training_curves(
    train_losses: List[float],
    val_losses: Optional[List[float]] = None,
    train_metrics: Optional[Dict[str, List[float]]] = None,
    val_metrics: Optional[Dict[str, List[float]]] = None,
    save_path: Optional[str] = None,
) -> plt.Figure:
    """
    Plot training and validation loss/metric curves.

    Args:
        train_losses: Training loss per epoch.
        val_losses: Validation loss per epoch.
        train_metrics: Dict of metric_name -> values per epoch.
        val_metrics: Dict of metric_name -> values per epoch.
        save_path: Optional path to save.

    Returns:
        matplotlib Figure.
    """
    num_plots = 1
    if train_metrics:
        num_plots += len(train_metrics)

    fig, axes = plt.subplots(1, num_plots, figsize=(6 * num_plots, 5))
    if num_plots == 1:
        axes = [axes]

    # Loss curve
    ax = axes[0]
    epochs = range(1, len(train_losses) + 1)
    ax.plot(epochs, train_losses, "b-", label="Train Loss", linewidth=2)
    if val_losses:
        ax.plot(epochs, val_losses, "r-", label="Val Loss", linewidth=2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Metric curves
    if train_metrics:
        for i, (name, values) in enumerate(train_metrics.items()):
            ax = axes[i + 1]
            ax.plot(range(1, len(values) + 1), values, "b-", label=f"Train {name}", linewidth=2)
            if val_metrics and name in val_metrics:
                val_vals = val_metrics[name]
                ax.plot(range(1, len(val_vals) + 1), val_vals, "r-", label=f"Val {name}", linewidth=2)
            ax.set_xlabel("Epoch")
            ax.set_ylabel(name)
            ax.set_title(name)
            ax.legend()
            ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig


def plot_confusion_matrix(
    cm: np.ndarray,
    class_names: List[str],
    title: str = "Confusion Matrix",
    save_path: Optional[str] = None,
) -> plt.Figure:
    """Plot a confusion matrix with labels."""
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="True Label",
        xlabel="Predicted Label",
        title=title,
    )

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    # Text annotations
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
            )

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig


def plot_property_distributions(
    property_vectors: np.ndarray,
    property_names: List[str],
    class_labels: Optional[np.ndarray] = None,
    class_names: Optional[List[str]] = None,
    save_path: Optional[str] = None,
) -> plt.Figure:
    """
    Plot distribution histograms for each property dimension.

    Args:
        property_vectors: Array of shape (N, num_properties).
        property_names: Names for each property dimension.
        class_labels: Optional integer class labels for coloring.
        class_names: Optional class name strings.
        save_path: Optional save path.
    """
    num_props = len(property_names)
    cols = 4
    rows = (num_props + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
    axes = axes.flatten()

    for i, (name, ax) in enumerate(zip(property_names, axes)):
        if class_labels is not None and class_names is not None:
            for cls_idx, cls_name in enumerate(class_names):
                mask = class_labels == cls_idx
                if mask.any():
                    ax.hist(property_vectors[mask, i], bins=30, alpha=0.6, label=cls_name)
            ax.legend(fontsize=6)
        else:
            ax.hist(property_vectors[:, i], bins=30, alpha=0.7, color="steelblue")

        ax.set_title(name, fontsize=9)
        ax.set_xlabel("Value", fontsize=7)
        ax.tick_params(labelsize=6)

    # Hide unused axes
    for j in range(num_props, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Property Vector Distributions", fontsize=14, fontweight="bold")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig



## 🚀 Execution Phase


In [ ]:
# 1. Organize Dataset
from collections import defaultdict

print('\n📦 Checking dataset structure...')
sixray_dir = os.path.join(RAW_DIR, 'sixray')
img_dir = os.path.join(PROCESSED_DIR, 'images')
label_dir = os.path.join(PROCESSED_DIR, 'labels')
os.makedirs(img_dir, exist_ok=True)
os.makedirs(label_dir, exist_ok=True)

if os.path.isdir(sixray_dir) and os.listdir(sixray_dir):
    is_roboflow = any(os.path.isdir(os.path.join(sixray_dir, split)) for split in ['train', 'valid', 'test'])
    if is_roboflow:
        print('Found Roboflow pre-split format. Flattening into processed directory...')
        for split in ['train', 'valid', 'test', 'val']:
            split_dir = os.path.join(sixray_dir, split)
            if not os.path.isdir(split_dir): continue
            split_img_dir = os.path.join(split_dir, 'images')
            if os.path.isdir(split_img_dir):
                for f in os.listdir(split_img_dir):
                    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                        shutil.copy2(os.path.join(split_img_dir, f), os.path.join(img_dir, f))
            split_label_dir = os.path.join(split_dir, 'labels')
            if os.path.isdir(split_label_dir):
                for f in os.listdir(split_label_dir):
                    if f.endswith('.txt'):
                        shutil.copy2(os.path.join(split_label_dir, f), os.path.join(label_dir, f))
stats = validate_dataset(PROCESSED_DIR)
assert stats['total_images'] > 0, 'No images found. Please upload dataset.'
print(f'\u2713 Dataset ready: {stats["total_images"]} images')



In [ ]:
# 2. Compute Properties
annotator = PropertyAnnotator()
image_files = sorted(glob.glob(os.path.join(img_dir, '*.*')))
label_files = [os.path.join(label_dir, os.path.splitext(os.path.basename(f))[0] + '.txt') for f in image_files]
output_csv = os.path.join(DATA_ROOT, 'annotations', 'properties.csv')
os.makedirs(os.path.dirname(output_csv), exist_ok=True)
if not os.path.exists(output_csv):
    print(f'\U0001f52c Computing properties for {len(image_files)} images...')
    props_df = batch_annotate(image_files, label_files, output_csv, annotator)
else:
    print('Properties already computed.')



In [ ]:
# 3. Create Stratified Splits
split_dir = os.path.join(DATA_ROOT, 'splits')
data_yaml = os.path.join(DATA_ROOT, 'data.yaml')
train_files, val_files, test_files = create_splits(
    img_dir, label_dir, split_dir,
    train_ratio=config['dataset']['train_ratio'],
    val_ratio=config['dataset']['val_ratio'],
    test_ratio=config['dataset']['test_ratio'],
    create_symlinks=True
)
create_yolo_data_yaml(split_dir, config['dataset']['threat_classes'], data_yaml)



In [ ]:
# 4. Initialize Model & Dataloaders
config['data_yaml_path'] = data_yaml
property_csv = os.path.join(DATA_ROOT, 'annotations', 'properties.csv')
train_dataset = XRayDataset(
    image_dir=os.path.join(split_dir, 'train', 'images'),
    label_dir=os.path.join(split_dir, 'train', 'labels'),
    property_csv=property_csv,
    image_size=config['dataset']['image_size']
)
val_dataset = XRayDataset(
    image_dir=os.path.join(split_dir, 'val', 'images'),
    label_dir=os.path.join(split_dir, 'val', 'labels'),
    property_csv=property_csv,
    image_size=config['dataset']['image_size']
)
def collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if not batch: return None
    images = torch.stack([item['image'] for item in batch])
    boxes = [item['boxes'] for item in batch]
    labels = [item['labels'] for item in batch]
    properties = torch.stack([item['properties'] for item in batch])
    materials = torch.stack([item['materials'] for item in batch])
    img_paths = [item['img_path'] for item in batch]
    return {'image': images, 'boxes': boxes, 'labels': labels, 'properties': properties, 'materials': materials, 'img_path': img_paths}

train_loader = DataLoader(train_dataset, batch_size=config['model1']['stage2']['batch_size'], shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=config['model1']['stage2']['batch_size'], shuffle=False, collate_fn=collate_fn)

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
model = PropertyYOLO(
    model_size=config['model1']['backbone'],
    num_classes=len(config['dataset']['threat_classes']) + 1,
    num_properties=config['model1']['property_head']['num_outputs'],
    input_channels=config['model1']['input_channels'],
    pretrained=config['model1']['pretrained']
)
trainer = Trainer(config=config, model=model, device=device, checkpoint_dir=CHECKPOINT_DIR)



In [ ]:
# 5. Train Stage 1 (Detection)
print('\nStarting Stage 1: Detection Training...')
stage1_results = trainer.train_stage1(train_loader=train_loader, val_loader=val_loader)



In [ ]:
# 6. Train Stage 2 (Property Regression)
print('\nStarting Stage 2: Property & Material Training...')
stage2_results = trainer.train_stage2(train_loader=train_loader, val_loader=val_loader)

